# 🛡️ MPLADS AI Risk Intelligence Engine — Colab (Part 1: Data + AI)

**SIH Problem Statement:** Build an AI-powered risk-detection and decision-support
system that examines MPLADS project, expenditure and progress data and tells
officials **which works look risky**, **why**, and **what to check first**.

### What this notebook does
This is the **offline "brain"** of the prototype. It does NOT rebuild the
official MPLADS/eSAKSHI dashboard — it reads data that mirrors what that
dashboard already exposes, and adds a new **AI Risk Intelligence layer** on
top of it:

1. Load the 4 MPLADS extract files (completed works, recommended works,
   expenditures, MP-wise summary)
2. Engineer risk *signals* (cost outliers, missing evidence, work
   fragmentation/splitting, fund-utilisation vs completion mismatches,
   vendor concentration, etc.)
3. Score every work and every MP with a **hybrid model**:
   - a transparent **rule-based score** (so officials can trust *why* something
     is flagged — no black box), **plus**
   - an **Isolation Forest** anomaly-detection layer (unsupervised ML, catches
     subtle combinations the fixed rules might miss)
4. Generate a plain-language **"why flagged"** explanation and a
   **"what to verify first"** checklist for every flagged item
5. Export everything as JSON — this is what **Part 2 (the Streamlit app)**
   will read and display to officials
6. Leave a clearly marked **API integration slot** for later, for when you
   plug this into the live eSAKSHI/MPLADS API instead of static CSV extracts

> ⚠️ **Important framing, baked into the design:** nothing in this notebook
> concludes that a work is fraudulent. Every score is a *"this pattern is
> unusual — a human should look at it"* signal, never a verdict.

---
**Run order:** run the cells top to bottom. Section 9 is the only part you'll
need to revisit once you have a live API.


## 0. Setup — install & import libraries

In [ ]:
# Everything here ships with Colab by default except nothing extra is needed —
# pandas / numpy / scikit-learn are all pre-installed on Colab runtimes.
# This cell is here so the notebook is self-contained even if you run it
# on a fresh runtime.
!pip install -q scikit-learn pandas numpy


In [ ]:
import pandas as pd
import numpy as np
import re
import json
from datetime import datetime, timezone

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_colwidth', 120)
print("Libraries loaded ✅")


## 1. Load the MPLADS data extracts

You need 4 files (same column layout as the official eSAKSHI/MPLADS exports):

| File | What it contains |
|---|---|
| `mplads_completed_works.csv` | Every work marked complete: amount, dates, whether site photos exist |
| `mplads_recommended_works.csv` | Every work an MP has recommended (sanctioned pipeline) |
| `mplads_expenditures.csv` | Every payment transaction: vendor, amount, payment status |
| `mplads_mp_summary.csv` | One row per MP: allocation, utilisation %, completion %, pending payments |

Run the cell below — it opens Colab's file picker. If you're using Google
Drive instead, see the commented alternative just under it.


In [ ]:
from google.colab import files

print("Upload the 4 MPLADS CSV files (you can select all 4 at once):")
uploaded = files.upload()   # opens a file picker in the Colab UI


In [ ]:
# --- Alternative: mount Google Drive instead of uploading each time ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/mplads_data'   # <- change to your folder
# Then skip the files.upload() cell above.

import glob

def find_file(keyword):
    """Finds the uploaded file whose name contains `keyword`, case-insensitive."""
    matches = [f for f in uploaded.keys() if keyword.lower() in f.lower()]
    if not matches:
        raise FileNotFoundError(f"No uploaded file matched '{keyword}'. "
                                 f"Files uploaded: {list(uploaded.keys())}")
    return matches[0]

completed_path   = find_file('completed')
recommended_path = find_file('recommended')
expenditure_path = find_file('expenditure')
mp_summary_path  = find_file('mp_summary')

print("Matched files:")
print(" completed  :", completed_path)
print(" recommended:", recommended_path)
print(" expenditure:", expenditure_path)
print(" mp_summary :", mp_summary_path)


In [ ]:
comp = pd.read_csv(completed_path)
rec  = pd.read_csv(recommended_path)
exp  = pd.read_csv(expenditure_path)
mps  = pd.read_csv(mp_summary_path)

# Strip stray whitespace from column headers (CSV exports sometimes have this)
for df in (comp, rec, exp, mps):
    df.columns = [c.strip() for c in df.columns]

print("Completed works :", comp.shape)
print("Recommended works:", rec.shape)
print("Expenditures     :", exp.shape)
print("MP summary       :", mps.shape)
comp.head(3)


## 2. Classify each work into a "work type"

The raw `Category` column in the data is almost always `"Normal/Others"` —
not useful on its own. To compare "is this cost normal?" we first need to
know *what kind* of work it is (a road costs differently from a solar
lightor a community hall). We do this with a transparent keyword classifier
— no ML needed here, and it's easy for an official to audit.


In [ ]:
WORK_TYPE_KEYWORDS = {
    'road_transport'     : r'\b(road|street|pathway|bridge|culvert|footpath|highway)\b',
    'drainage'           : r'\b(drain|drainage|sewer|sewerage)\b',
    'drinking_water'     : r'\b(drinking water|water supply|bore ?well|hand pump|overhead tank|pipeline)\b',
    'sanitation'         : r'\b(toilet|sanitation|washroom|urinal)\b',
    'education'          : r'\b(school|vidyalaya|classroom|anganwadi|college|library)\b',
    'health'             : r'\b(hospital|health.?center|health.?centre|phc|dispensary|ambulance|medical)\b',
    'community_hall'     : r'\b(community (center|centre|hall)|panchayat bhawan|marriage hall|kalyan mandap|assembly hall)\b',
    'solar_lighting'     : r'\b(solar|street light|led light|high mast)\b',
    'sports'             : r'\b(playground|stadium|sports|gym)\b',
    'furniture_equipment': r'\b(furniture|desk|bench|chair|table|tricycle|wheelchair|equipment)\b',
    'electricity'        : r'\b(electric|transformer|substation)\b',
    'crematorium'        : r'\b(crematorium|shamshan|graveyard|cemetery)\b',
    'agriculture'        : r'\b(irrigation|canal|agricultur|farm)\b',
}

def classify_work_type(description: str) -> str:
    """Returns the first matching work-type keyword bucket, else 'other'."""
    text = str(description).lower()
    for label, pattern in WORK_TYPE_KEYWORDS.items():
        if re.search(pattern, text):
            return label
    return 'other'


def normalize_description(description: str) -> str:
    """Strips directional/serial words (NORTH/SOUTH/PART/PHASE/numbers) so that
    'X North Part' and 'X South Part' collapse to the same normalized text.
    This is what lets us later detect a single work artificially split into
    many near-identical smaller works."""
    text = str(description).lower()
    text = re.sub(r'\b(north|south|east|west|part|phase|reach|zone|block|sector|no\.?\s*\d+|\d+)\b', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Classifier ready ✅")


## 3. Build one unified "works" table

We combine `completed_works` and `recommended_works` into a single table so
every project — finished or still pending — gets a risk score.


In [ ]:
comp2 = comp.rename(columns={'Final Amount (₹)': 'amount', 'Completed Date': 'date'}).copy()
comp2['status'] = 'Completed'
comp2['has_images'] = comp2['Has Images'].fillna(False)

rec2 = rec.rename(columns={'Recommended Amount (₹)': 'amount', 'Recommendation Date': 'date'}).copy()
rec2['status'] = 'Recommended (pending)'
rec2['has_images'] = rec2['Has Images'].fillna(False)

KEEP_COLS = ['Work ID', 'Work Description', 'MP Name', 'Constituency', 'State',
             'House', 'amount', 'date', 'has_images', 'status', 'IDA']

works = pd.concat([comp2[KEEP_COLS], rec2[KEEP_COLS]], ignore_index=True)

# Clean amounts
works['amount'] = pd.to_numeric(works['amount'], errors='coerce')
works = works.dropna(subset=['amount'])
works = works[works['amount'] > 0].reset_index(drop=True)

# Apply the classifiers from Section 2
works['work_type'] = works['Work Description'].apply(classify_work_type)
works['norm_desc']  = works['Work Description'].apply(normalize_description)
works['log_amount'] = np.log1p(works['amount'])

print("Unified works table:", works.shape)
works[['Work Description', 'work_type', 'amount', 'status']].sample(5, random_state=1)


## 4. Engineer work-level risk signals

Four independent, explainable signals. Each is a simple yes/no rule so it
can be explained in one sentence to a non-technical official.

| Signal | What it catches |
|---|---|
| **Cost outlier** | Work costs far more (or less) than similar works of the same type, nationwide |
| **Round amount** | Amount is a suspiciously exact round figure (e.g. exactly ₹5,00,000) |
| **Missing evidence** | Marked *Completed* but has no site photo on record |
| **Fragmentation** | Same MP has 5+ near-identical works at an identical amount — possible work-splitting to dodge approval thresholds, OR a legitimate bulk scheme (e.g. desks to 50 schools) that still deserves a one-line sanity check |


In [ ]:
# --- 4a. Cost outlier: robust z-score of log(amount) within each work_type ---
grp = works.groupby('work_type')['log_amount']
median = grp.transform('median')
q1 = grp.transform(lambda s: s.quantile(0.25))
q3 = grp.transform(lambda s: s.quantile(0.75))
iqr = (q3 - q1).replace(0, np.nan)

works['cost_robust_z'] = ((works['log_amount'] - median) / iqr).fillna(0)
works['flag_cost_outlier'] = works['cost_robust_z'].abs() > 2.5


In [ ]:
# --- 4b. Suspiciously round amounts (₹1,00,000 multiples, above ₹1 lakh) ---
works['flag_round_amount'] = (works['amount'] % 100_000 == 0) & (works['amount'] >= 100_000)


In [ ]:
# --- 4c. Missing evidence: only meaningful for COMPLETED works ---
works['flag_missing_evidence'] = (works['status'] == 'Completed') & (~works['has_images'])


In [ ]:
# --- 4d. Fragmentation / possible work-splitting ---
# Group by (MP, normalized description, exact amount). If 5+ works share all
# three, they are near-identical works billed at an identical figure.
fragment_groups = (
    works.groupby(['MP Name', 'norm_desc', 'amount'])
         .size()
         .reset_index(name='cluster_size')
)
fragment_groups = fragment_groups[
    (fragment_groups['norm_desc'].str.len() > 3) & (fragment_groups['cluster_size'] >= 5)
]
cluster_lookup = fragment_groups.set_index(['MP Name', 'norm_desc', 'amount'])['cluster_size'].to_dict()

works['fragmentation_cluster_size'] = (
    works.set_index(['MP Name', 'norm_desc', 'amount']).index.map(cluster_lookup).fillna(0)
)
works['flag_fragmentation'] = works['fragmentation_cluster_size'] >= 5

print("Flag rates:")
for col in ['flag_cost_outlier', 'flag_round_amount', 'flag_missing_evidence', 'flag_fragmentation']:
    print(f"  {col:<24s}: {works[col].mean()*100:5.2f}%  ({works[col].sum()} works)")


## 5. Transparent rule-based risk score (0–100)

This is the **explainable layer**. Each flag contributes fixed points.
Officials can see exactly why a number is what it is — nothing hidden.
Feel free to tune these weights based on domain-expert feedback during
the hackathon judging / pilot.


In [ ]:
RULE_WEIGHTS = {
    'flag_cost_outlier'    : 30,
    'flag_round_amount'    : 10,
    'flag_missing_evidence': 20,
    'flag_fragmentation'   : 40,   # weighted highest: pattern most associated with fund misuse
}

works['rule_score'] = sum(works[flag].astype(int) * weight for flag, weight in RULE_WEIGHTS.items())
works['rule_score'] = works['rule_score'].clip(0, 100)

works['rule_score'].describe()


## 6. AI layer — Isolation Forest anomaly detection

The rules above are hand-crafted. To catch anomalies that don't fit a
predefined rule (unusual *combinations* of otherwise-normal-looking values),
we run an **Isolation Forest** — an unsupervised model built exactly for
"find the unusual points in this data" problems, and it doesn't need any
labelled fraud examples (which we don't have).


In [ ]:
FEATURE_COLS = ['log_amount', 'cost_robust_z', 'fragmentation_cluster_size']

X = works[FEATURE_COLS].fillna(0).values
X_scaled = StandardScaler().fit_transform(X)

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.08,   # assume ~8% of works look anomalous — tune this during the pilot
    random_state=42,
)
iso_forest.fit(X_scaled)

# score_samples: higher = more "normal". We flip the sign so higher = more anomalous.
raw_anomaly = -iso_forest.score_samples(X_scaled)
works['ml_anomaly_score'] = (
    (raw_anomaly - raw_anomaly.min()) / (raw_anomaly.max() - raw_anomaly.min()) * 100
)

works['ml_anomaly_score'].describe()


## 7. Combine into one final risk score

`final_risk_score = 0.6 × rule_score + 0.4 × ML anomaly_score`

The rule-based half keeps the score explainable; the ML half adds sensitivity
to subtler anomalies. 60/40 is a starting point — a real deployment would
tune this weighting against outcomes from actual field verifications.


In [ ]:
works['final_risk_score'] = (0.6 * works['rule_score'] + 0.4 * works['ml_anomaly_score']).round(1)

def risk_band(score):
    if score >= 60:
        return 'High'
    elif score >= 30:
        return 'Medium'
    return 'Low'

works['risk_band'] = works['final_risk_score'].apply(risk_band)

works['risk_band'].value_counts()


## 8. Explainability — "why is this flagged?" + "what to check first?"

For every work, turn the flags that fired into **plain-English reasons**
and a matching **verification checklist**. This directly answers SIH
questions 2 and 3 ("why risky" / "what to verify").


In [ ]:
REASON_TEXT = {
    'flag_cost_outlier'    : "Cost is a statistical outlier vs similar '{wt}' works nationwide",
    'flag_round_amount'    : "Amount is an unusually round figure (₹{amt:,.0f})",
    'flag_missing_evidence': "Marked completed but has no supporting site photos uploaded",
    'flag_fragmentation'   : "Part of a cluster of {n} near-identical works by the same MP (possible work-splitting)",
}
VERIFY_TEXT = {
    'flag_cost_outlier'    : "Compare against the state Schedule of Rates (SOR) for this work category",
    'flag_round_amount'    : "Cross-check the vendor quotation/estimate backing this exact figure",
    'flag_missing_evidence': "Request geo-tagged site photos or schedule a physical inspection",
    'flag_fragmentation'   : "Confirm whether this is a legitimate sanctioned batch scheme, or works were split to avoid approval/tender thresholds",
}

def build_reasons(row):
    reasons, checklist = [], []
    if row['flag_cost_outlier']:
        reasons.append(REASON_TEXT['flag_cost_outlier'].format(wt=row['work_type'].replace('_', ' ')))
        checklist.append(VERIFY_TEXT['flag_cost_outlier'])
    if row['flag_round_amount']:
        reasons.append(REASON_TEXT['flag_round_amount'].format(amt=row['amount']))
        checklist.append(VERIFY_TEXT['flag_round_amount'])
    if row['flag_missing_evidence']:
        reasons.append(REASON_TEXT['flag_missing_evidence'])
        checklist.append(VERIFY_TEXT['flag_missing_evidence'])
    if row['flag_fragmentation']:
        reasons.append(REASON_TEXT['flag_fragmentation'].format(n=int(row['fragmentation_cluster_size'])))
        checklist.append(VERIFY_TEXT['flag_fragmentation'])
    if not reasons:
        reasons = ["No strong anomaly signals detected"]
        checklist = ["Routine monitoring is sufficient"]
    return pd.Series([reasons, checklist])

works[['reasons', 'verify_checklist']] = works.apply(build_reasons, axis=1)

# Peek at a flagged example
works[works['risk_band'] == 'High'][
    ['Work Description', 'MP Name', 'final_risk_score', 'reasons', 'verify_checklist']
].head(3)


## 9. MP / fund-level risk layer

Work-level scoring above catches *individual project* red flags. This
section adds a **second, independent layer** using `mp_summary` and
`expenditures` to catch *fund-management* red flags at the MP level:

- **Utilisation–completion mismatch**: money is recorded as spent, but very
  few works are actually complete on the ground
- **Vendor concentration**: one vendor receiving a disproportionate share of
  an MP's spending
- **High unpaid balance**: large sums sanctioned but never paid out


In [ ]:
mps['utilization_completion_gap'] = mps['Utilization %'] - mps['Completion Rate %']
mps['pending_ratio'] = mps['Pending Payments'] / mps['Transaction Count'].replace(0, np.nan)
mps['balance_ratio'] = (
    mps['Balance Not Yet Paid to Vendors (₹)'] / mps['Allocated Amount (₹)'].replace(0, np.nan)
)

# Vendor concentration per MP (Herfindahl-Hirschman Index — higher = more concentrated)
vendor_amt = exp.groupby(['MP Name', 'Vendor'])['Expenditure Amount (₹)'].sum().reset_index()
mp_total = vendor_amt.groupby('MP Name')['Expenditure Amount (₹)'].transform('sum')
vendor_amt['share'] = vendor_amt['Expenditure Amount (₹)'] / mp_total

hhi = vendor_amt.groupby('MP Name')['share'].apply(lambda s: (s ** 2).sum()).reset_index(name='vendor_hhi')
top_vendor = (
    vendor_amt.sort_values('share', ascending=False)
    .drop_duplicates('MP Name')[['MP Name', 'Vendor', 'share']]
)
top_vendor.columns = ['MP Name', 'top_vendor', 'top_vendor_share']

mps = mps.merge(hhi, on='MP Name', how='left').merge(top_vendor, on='MP Name', how='left')
mps['vendor_hhi'] = mps['vendor_hhi'].fillna(0)
mps['top_vendor_share'] = mps['top_vendor_share'].fillna(0)

mps['flag_util_completion_mismatch'] = mps['utilization_completion_gap'] >= 60
mps['flag_vendor_concentration'] = mps['top_vendor_share'] >= 0.4
mps['flag_high_pending_balance'] = mps['balance_ratio'] >= 0.5

print("MP-level flag rates:")
for col in ['flag_util_completion_mismatch', 'flag_vendor_concentration', 'flag_high_pending_balance']:
    print(f"  {col:<32s}: {mps[col].mean()*100:5.2f}%  ({mps[col].sum()} MPs)")


In [ ]:
MP_WEIGHTS = {
    'flag_util_completion_mismatch': 45,
    'flag_vendor_concentration'    : 30,
    'flag_high_pending_balance'    : 25,
}
mps['mp_rule_score'] = sum(mps[flag].astype(int) * weight for flag, weight in MP_WEIGHTS.items())

MP_FEATURE_COLS = ['utilization_completion_gap', 'vendor_hhi', 'balance_ratio']
Xm = mps[MP_FEATURE_COLS].fillna(0).values
Xm_scaled = StandardScaler().fit_transform(Xm)

iso_forest_mp = IsolationForest(n_estimators=200, contamination=0.1, random_state=42)
iso_forest_mp.fit(Xm_scaled)
raw_mp_anomaly = -iso_forest_mp.score_samples(Xm_scaled)
mps['mp_ml_score'] = (raw_mp_anomaly - raw_mp_anomaly.min()) / (raw_mp_anomaly.max() - raw_mp_anomaly.min()) * 100

mps['mp_final_risk_score'] = (0.6 * mps['mp_rule_score'] + 0.4 * mps['mp_ml_score']).round(1)
mps['mp_risk_band'] = mps['mp_final_risk_score'].apply(risk_band)

mps['mp_risk_band'].value_counts()


In [ ]:
MP_REASON_TEXT = {
    'flag_util_completion_mismatch': "{u:.0f}% of funds recorded as utilised but only {c:.0f}% of works are physically completed",
    'flag_vendor_concentration'    : "{share:.0f}% of this MP's expenditure has gone to a single vendor ({vendor})",
    'flag_high_pending_balance'    : "₹{bal:,.0f} allocated but still not paid out to any vendor",
}
MP_VERIFY_TEXT = {
    'flag_util_completion_mismatch': "Field-verify physical progress of works marked 'in progress' against reported spend",
    'flag_vendor_concentration'    : "Check vendor empanelment process and look for related-party or single-bidder tenders",
    'flag_high_pending_balance'    : "Review why sanctioned works have not moved to payment stage",
}

def build_mp_reasons(row):
    reasons, checklist = [], []
    if row['flag_util_completion_mismatch']:
        reasons.append(MP_REASON_TEXT['flag_util_completion_mismatch'].format(
            u=row['Utilization %'], c=row['Completion Rate %']))
        checklist.append(MP_VERIFY_TEXT['flag_util_completion_mismatch'])
    if row['flag_vendor_concentration']:
        reasons.append(MP_REASON_TEXT['flag_vendor_concentration'].format(
            share=row['top_vendor_share'] * 100, vendor=row['top_vendor']))
        checklist.append(MP_VERIFY_TEXT['flag_vendor_concentration'])
    if row['flag_high_pending_balance']:
        reasons.append(MP_REASON_TEXT['flag_high_pending_balance'].format(
            bal=row['Balance Not Yet Paid to Vendors (₹)']))
        checklist.append(MP_VERIFY_TEXT['flag_high_pending_balance'])
    if not reasons:
        reasons = ["No strong anomaly signals detected"]
        checklist = ["Routine monitoring is sufficient"]
    return pd.Series([reasons, checklist])

mps[['reasons', 'verify_checklist']] = mps.apply(build_mp_reasons, axis=1)
mps[mps['mp_risk_band'] == 'High'][['MP Name', 'State', 'mp_final_risk_score', 'reasons']].head(3)


## 10. Export results for the Streamlit app

The Streamlit app (Part 2, a **separate file** — `app.py`) never touches
pandas/sklearn directly. It just reads the JSON files this cell produces.
This keeps the two halves of the system cleanly separated:

- **Colab = the AI engine** (this notebook): heavy lifting, scoring, retrainable
- **Streamlit = the decision-support UI**: fast, lightweight, official-facing

Download the files this cell produces and drop them into the
`streamlit_app/data/` folder next to `app.py`.


In [ ]:
import os
os.makedirs('mplads_ai_outputs', exist_ok=True)

# Full scored tables, sorted riskiest-first
works_out = works[[
    'Work ID', 'Work Description', 'MP Name', 'Constituency', 'State', 'House',
    'work_type', 'amount', 'date', 'status', 'has_images',
    'final_risk_score', 'risk_band', 'reasons', 'verify_checklist',
    'flag_cost_outlier', 'flag_round_amount', 'flag_missing_evidence', 'flag_fragmentation',
]].sort_values('final_risk_score', ascending=False)
works_out['Work ID'] = works_out['Work ID'].astype(int)

mps_out = mps[[
    'MP Name', 'Constituency', 'State', 'House',
    'Allocated Amount (₹)', 'Total Expenditure (₹)', 'Utilization %', 'Completion Rate %',
    'Completed Works', 'Recommended Works', 'Balance Not Yet Paid to Vendors (₹)',
    'top_vendor', 'top_vendor_share', 'mp_final_risk_score', 'mp_risk_band',
    'reasons', 'verify_checklist',
    'flag_util_completion_mismatch', 'flag_vendor_concentration', 'flag_high_pending_balance',
]].sort_values('mp_final_risk_score', ascending=False)

# The dashboard's "priority queue" only needs the top N riskiest works —
# officials want a short actionable list, not all ~130k rows.
TOP_N = 3000
works_out.head(TOP_N).to_json(f'mplads_ai_outputs/risk_scores_works.json', orient='records')
mps_out.to_json(f'mplads_ai_outputs/risk_scores_mps.json', orient='records')

# Pre-aggregated numbers for charts, computed on the FULL scored set
state_agg = (
    works_out.groupby('State')
    .agg(total_works=('final_risk_score', 'size'),
         high_risk=('risk_band', lambda s: (s == 'High').sum()),
         avg_risk=('final_risk_score', 'mean'))
    .reset_index().sort_values('high_risk', ascending=False)
)
worktype_agg = (
    works_out.groupby('work_type')
    .agg(total_works=('final_risk_score', 'size'),
         high_risk=('risk_band', lambda s: (s == 'High').sum()),
         avg_risk=('final_risk_score', 'mean'))
    .reset_index().sort_values('high_risk', ascending=False)
)
state_agg.to_json('mplads_ai_outputs/aggregate_by_state.json', orient='records')
worktype_agg.to_json('mplads_ai_outputs/aggregate_by_worktype.json', orient='records')

meta = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "total_works_scored": int(len(works_out)),
    "total_mps_scored": int(len(mps_out)),
    "high_risk_works": int((works_out['risk_band'] == 'High').sum()),
    "medium_risk_works": int((works_out['risk_band'] == 'Medium').sum()),
    "high_risk_mps": int((mps_out['mp_risk_band'] == 'High').sum()),
    "priority_queue_size": int(min(TOP_N, len(works_out))),
}
with open('mplads_ai_outputs/meta_summary.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(json.dumps(meta, indent=2))


In [ ]:
# Download everything as one zip so you can hand it straight to the Streamlit app
import shutil
shutil.make_archive('mplads_ai_outputs', 'zip', 'mplads_ai_outputs')
files.download('mplads_ai_outputs.zip')


## 11. 🔌 API integration slot (fill in later)

Right now, Section 1 reads static CSV exports. When you're ready to connect
this to a **live API** (the official eSAKSHI/MPLADS API, or your own backend
that mirrors it), you only need to touch **this section** — nothing above it
changes, because everything above works off the `comp` / `rec` / `exp` /
`mps` DataFrames regardless of where they came from.

Replace the placeholder function below, then re-run Sections 3–10.


In [ ]:
import requests

def fetch_live_mplads_data(api_base_url: str, api_key: str = None):
    """
    TODO: point this at the real API once you have the endpoint details.

    Expected to return the same 4 DataFrames as Section 1:
        completed_df, recommended_df, expenditure_df, mp_summary_df

    Example shape once you have real endpoints (edit to match the actual API):

        headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}

        completed_df   = pd.DataFrame(requests.get(f"{api_base_url}/works/completed",   headers=headers).json())
        recommended_df = pd.DataFrame(requests.get(f"{api_base_url}/works/recommended", headers=headers).json())
        expenditure_df = pd.DataFrame(requests.get(f"{api_base_url}/expenditures",      headers=headers).json())
        mp_summary_df  = pd.DataFrame(requests.get(f"{api_base_url}/mps/summary",       headers=headers).json())

        return completed_df, recommended_df, expenditure_df, mp_summary_df
    """
    raise NotImplementedError("Plug in the real API details here, then re-run from Section 3 onward.")


# --- Switch used at the top of the notebook ---
USE_LIVE_API = False   # flip to True once fetch_live_mplads_data() is implemented

if USE_LIVE_API:
    comp, rec, exp, mps = fetch_live_mplads_data(
        api_base_url="https://YOUR-API-HOST/api/v1",
        api_key="YOUR_API_KEY",
    )
    print("Loaded live data from API ✅ — now re-run Sections 3 through 10.")
else:
    print("Still in static-CSV mode. Set USE_LIVE_API = True once the API is ready.")
